# Sesión de clase 12 · Evidencia para el informe CRISP-ML(Q) y servidor MCP

Este notebook genera las cifras del informe
*«Plantilla informe - CRISP-ML(Q) y servidor MCP»*:

- **Parte A, sección 3:** informe de calidad de datos de `productos.csv`,
  `clientes.csv` y del historial (`resenas_entrega.csv`).
- **Parte A, secciones 4–5:** evaluación offline del prototipo de
  recomendador semántico (laboratorio de bases de datos vectoriales, sesión 06)
  frente a dos líneas base, revisión manual de una muestra y pruebas de robustez.
- **Parte B, sección 9:** evaluación de la recuperación de las herramientas del
  servidor MCP, llamándolas directamente (sin LLM).

El modelo de embeddings es el del curso, `paraphrase-multilingual-MiniLM-L12-v2`
(384 dimensiones), y se ejecuta localmente en CPU. En lugar de ChromaDB se usa
similitud coseno con NumPy: con 90 productos el resultado es el mismo y el
notebook no depende de una colección persistida.

**Etapas:** 1. Cargar · 2. Calidad de datos · 3. Prototipo del recomendador ·
4. Evaluación offline · 5. Revisión manual · 6. Robustez · 7. Herramientas MCP ·
8. Evaluación de la recuperación.

In [1]:
import random
import re

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)

SEMILLA = 42
MODELO = "paraphrase-multilingual-MiniLM-L12-v2"

## 1. Cargar los datos

In [2]:
productos = pd.read_csv("../data/productos.csv")
resenas = pd.read_csv("../data/resenas_entrega.csv")
clientes = pd.read_csv("../data/clientes.csv")
comentarios = pd.read_csv("../data/comentarios.csv")

for nombre, df in [("productos", productos), ("resenas_entrega", resenas),
                   ("clientes", clientes), ("comentarios", comentarios)]:
    print(f"{nombre:16s} {df.shape}  columnas: {list(df.columns)}")

productos        (90, 11)  columnas: ['id', 'codigo', 'nombre', 'descripcion', 'categoria', 'subcategoria', 'precio', 'stock', 'calificacion_promedio', 'total_resenas', 'url']
resenas_entrega  (110, 7)  columnas: ['id', 'orden_id', 'cliente_id', 'producto_id', 'calificacion', 'fecha', 'texto']
clientes         (120, 7)  columnas: ['id', 'dni', 'nombre', 'apellidos', 'email', 'ciudad', 'pais']
comentarios      (135, 8)  columnas: ['id', 'cliente_id', 'cliente_nombre_completo', 'cliente_ciudad', 'cliente_pais', 'calificacion', 'fecha', 'texto']


## 2. Informe de calidad de datos (sección 3 del informe)

### 2.1 Clientes con historial y reseñas por cliente

In [3]:
por_cliente = resenas["cliente_id"].value_counts()
n_con = por_cliente.size
print(f"Clientes con al menos una reseña: {n_con} / {len(clientes)} ({n_con / len(clientes):.1%})")
sin_historial = sorted(set(clientes["id"]) - set(resenas["cliente_id"]))
print(f"Clientes sin reseñas: {len(sin_historial)} (ejemplos: {sin_historial[:5]})")
print("Distribución (n.º de reseñas -> n.º de clientes):", por_cliente.value_counts().sort_index().to_dict())
print(f"Mediana {por_cliente.median():.0f} · media {por_cliente.mean():.2f} · máximo {por_cliente.max()} (cliente {por_cliente.idxmax()})")
print("Clientes con >= 2 reseñas:", int((por_cliente >= 2).sum()))
print("Pares cliente-producto repetidos (recompras):", int(resenas.duplicated(["cliente_id", "producto_id"]).sum()))

Clientes con al menos una reseña: 63 / 120 (52.5%)
Clientes sin reseñas: 57 (ejemplos: [4, 5, 6, 10, 11])
Distribución (n.º de reseñas -> n.º de clientes): {1: 39, 2: 12, 3: 8, 4: 2, 5: 1, 10: 1}
Mediana 1 · media 1.75 · máximo 10 (cliente 1)
Clientes con >= 2 reseñas: 24
Pares cliente-producto repetidos (recompras): 9


### 2.2 Productos sin reseñas (arranque en frío) y distribución por categoría

In [4]:
sin_resenas = sorted(set(productos["id"]) - set(resenas["producto_id"]))
print(f"Productos sin reseñas: {len(sin_resenas)} / {len(productos)} ({len(sin_resenas) / len(productos):.0%})")
print("Ejemplos:", productos.set_index("id").loc[sin_resenas[:3], "nombre"].to_dict())
print("Por categoría:", productos[productos["id"].isin(sin_resenas)]["categoria"].value_counts().to_dict())

Productos sin reseñas: 36 / 90 (40%)
Ejemplos: {1: 'Apple iPhone 15 Pro Max 256GB', 2: 'Apple iPhone 15 128GB', 6: 'Xiaomi Poco X6 Pro 512GB'}
Por categoría: {'Smartphones': 4, 'Laptops y Computadoras': 4, 'Audio': 4, 'Televisores y Video': 4, 'Gaming y Consolas': 4, 'Wearables y Fitness': 4, 'Fotografia y Video': 4, 'Electrodomesticos Inteligentes': 4, 'Accesorios y Componentes': 4}


In [5]:
print("Productos por categoría:", productos["categoria"].value_counts().to_dict())
por_sub = productos.groupby(["categoria", "subcategoria"]).size()
print(f"Subcategorías: {por_sub.size} · productos por subcategoría: min {por_sub.min()}, max {por_sub.max()}")
por_sub.to_frame("n_productos")

Productos por categoría: {'Smartphones': 10, 'Laptops y Computadoras': 10, 'Audio': 10, 'Televisores y Video': 10, 'Gaming y Consolas': 10, 'Wearables y Fitness': 10, 'Fotografia y Video': 10, 'Electrodomesticos Inteligentes': 10, 'Accesorios y Componentes': 10}
Subcategorías: 34 · productos por subcategoría: min 1, max 6


n_productos
categoria                      subcategoria                         
Accesorios y Componentes       Almacenamiento                      4
                               Cables y Adaptadores                1
                               Cargadores y Baterias               2
                               Componentes PC                      1
                               Perifericos                         2
Audio                          Audifonos Inalambricos              6
                               Parlantes Portatiles                4
Electrodomesticos Inteligentes Asistentes Virtuales                2
                               Cocina Inteligente                  2
                               Iluminacion Inteligente             1
                               Linea Blanca Inteligente            1
                               Robots Aspiradora                   2
                               Seguridad Inteligente               2
Fotografia y Video             Camaras DSLR                        1
                               Camaras Instantaneas                1
                               Camaras Mirrorless                  3
                               Camaras de Accion                   3
                               Drones                              2
Gaming y Consolas              Consolas                            5
                               Perifericos Gaming                  5
Laptops y Computadoras         Gaming                              3
                               Ultrabooks                          3
                               Uso General                         3
                               Uso Profesional                     1
Smartphones                    Gama Alta                           5
                               Gama Basica                         2
                               Gama Media                          3
Televisores y Video            Televisores LED                     5
                               Televisores OLED                    2
                               Televisores QLED                    2
                               Televisores ULED                    1
Wearables y Fitness            Bandas de Actividad                 2
                               Relojes Deportivos                  3
                               Smartwatches                        5

### 2.3 Calidad de las descripciones

In [6]:
largo = productos["descripcion"].str.len()
palabras = productos["descripcion"].str.split().str.len()
print("Caracteres:", largo.describe()[["min", "50%", "max"]].to_dict())
print("Palabras:", palabras.describe()[["min", "50%", "max"]].to_dict())
print("Más corta: id", productos.loc[largo.idxmin(), "id"], "· más larga: id", productos.loc[largo.idxmax(), "id"])
print("Faltantes:", int(productos["descripcion"].isna().sum()),
      "· vacías:", int((productos["descripcion"].str.strip() == "").sum()),
      "· duplicadas:", int(productos["descripcion"].duplicated().sum()))
print("Descripciones con tildes o ñ:", int(productos["descripcion"].str.contains("[áéíóúñÁÉÍÓÚÑ]").sum()))
print("Stock: mínimo", productos["stock"].min(), "· productos con stock 0:", int((productos["stock"] == 0).sum()))

Caracteres: {'min': 148.0, '50%': 185.0, 'max': 263.0}
Palabras: {'min': 22.0, '50%': 29.5, 'max': 44.0}
Más corta: id 39 · más larga: id 1
Faltantes: 0 · vacías: 0 · duplicadas: 0
Descripciones con tildes o ñ: 0
Stock: mínimo 3 · productos con stock 0: 0


### 2.4 Consistencia de `total_resenas` y `calificacion_promedio` con el historial

In [7]:
hist = resenas.groupby("producto_id").agg(n_historial=("id", "size"), prom_historial=("calificacion", "mean"))
cons = productos.set_index("id")[["nombre", "total_resenas", "calificacion_promedio"]].join(hist)
cons["n_historial"] = cons["n_historial"].fillna(0).astype(int)
coincide_n = (cons["total_resenas"] == cons["n_historial"]).sum()
con_hist = cons[cons["n_historial"] > 0]
coincide_prom = ((con_hist["calificacion_promedio"] - con_hist["prom_historial"]).abs() <= 0.05).sum()
print(f"total_resenas coincide con el historial: {coincide_n} / {len(cons)}")
print(f"calificacion_promedio coincide (±0,05): {coincide_prom} / {len(con_hist)} productos con historial")
print("Productos sin historial pero con total_resenas > 0:",
      int(((cons["n_historial"] == 0) & (cons["total_resenas"] > 0)).sum()))
cons.loc[[3, 10]]

total_resenas coincide con el historial: 8 / 90
calificacion_promedio coincide (±0,05): 7 / 54 productos con historial
Productos sin historial pero con total_resenas > 0: 36


,nombre,total_resenas,calificacion_promedio,n_historial,prom_historial
id,,,,,
3,Samsung Galaxy S24 Ultra 512GB,4,3.3,4,5.0
10,OnePlus 12 256GB,4,3.5,2,1.0


Las columnas agregadas del catálogo provienen de otra fuente: en la tienda
(`lib/db/schema.sql`) se calculan con los comentarios de tipo `producto`, que no
están en los CSV del curso. `resenas_entrega.csv` solo contiene `post_compra`.

### 2.5 Datos personales

In [8]:
print("Columnas de clientes.csv:", list(clientes.columns))
print("Columnas personales en comentarios.csv:", [c for c in comentarios.columns if c.startswith("cliente")])
print("DNI duplicados:", int(clientes["dni"].duplicated().sum()), "· correos duplicados:", int(clientes["email"].duplicated().sum()))
patron_pii = r"@|\b\d{8}\b"
print("Textos con correo o número de 8 dígitos -> reseñas:", int(resenas["texto"].str.contains(patron_pii).sum()),
      "· comentarios:", int(comentarios["texto"].str.contains(patron_pii).sum()))

Columnas de clientes.csv: ['id', 'dni', 'nombre', 'apellidos', 'email', 'ciudad', 'pais']
Columnas personales en comentarios.csv: ['cliente_id', 'cliente_nombre_completo', 'cliente_ciudad', 'cliente_pais']
DNI duplicados: 0 · correos duplicados: 0
Textos con correo o número de 8 dígitos -> reseñas: 0 · comentarios: 0


## 3. Prototipo del recomendador

Igual que en el laboratorio de la sesión 06: texto del producto =
`nombre + categoría + descripción`; perfil del cliente = promedio de los
embeddings de sus productos ponderado por la calificación; se recomiendan los
cinco productos más cercanos que el cliente no ha comprado.

In [9]:
modelo = SentenceTransformer(MODELO)
texto_producto = productos["nombre"] + ". " + productos["categoria"] + ". " + productos["descripcion"]
E = modelo.encode(texto_producto.tolist(), normalize_embeddings=True)
ids = productos["id"].tolist()
pos = {pid: i for i, pid in enumerate(ids)}
categoria = dict(zip(productos["id"], productos["categoria"]))
nombre = dict(zip(productos["id"], productos["nombre"]))
E.shape

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

(90, 384)

In [10]:
def perfil(historial):
    pesos = historial["calificacion"].to_numpy(dtype=float)
    V = E[[pos[p] for p in historial["producto_id"]]]
    v = (pesos[:, None] * V).sum(axis=0) / pesos.sum()
    return v / np.linalg.norm(v)


def recomendar(historial, n=5, excluir_sin_stock=None):
    excluir = set(historial["producto_id"]) | set(excluir_sin_stock or [])
    similitud = E @ perfil(historial)
    return [ids[i] for i in np.argsort(-similitud) if ids[i] not in excluir][:n]


def recomendar_populares(entrenamiento, historial, n=5):
    conteo = entrenamiento["producto_id"].value_counts()
    orden = list(conteo.index) + [p for p in ids if p not in conteo.index]
    excluir = set(historial["producto_id"])
    return [p for p in orden if p not in excluir][:n]

## 4. Evaluación offline (sección 5 del informe)

**Partición leave-last-out temporal:** para cada cliente con al menos dos
reseñas se reserva la más reciente como prueba y el perfil se construye solo
con las anteriores. La línea base de popularidad también se calcula sin la
reseña retenida. Con un solo producto relevante por cliente,
recall@5 = tasa de acierto y precisión@5 = recall@5 / 5.

In [11]:
resenas_ord = resenas.sort_values(["fecha", "id"])
evaluables = por_cliente[por_cliente >= 2].index
rng = random.Random(SEMILLA)
filas, recompras = [], 0
for cid in evaluables:
    h = resenas_ord[resenas_ord["cliente_id"] == cid]
    prueba, entrenamiento = h.iloc[-1], h.iloc[:-1]
    recompras += prueba["producto_id"] in set(entrenamiento["producto_id"])
    candidatos = {
        "semantico": recomendar(entrenamiento),
        "popularidad": recomendar_populares(resenas_ord.drop(prueba.name), entrenamiento),
        "aleatorio": rng.sample([p for p in ids if p not in set(entrenamiento["producto_id"])], 5),
    }
    for metodo, recs in candidatos.items():
        acierto = prueba["producto_id"] in recs
        filas.append({
            "metodo": metodo,
            "recall@5": float(acierto),
            "mrr": 1 / (recs.index(prueba["producto_id"]) + 1) if acierto else 0.0,
            "acierto_categoria@5": float(categoria[prueba["producto_id"]] in {categoria[p] for p in recs}),
        })
evaluacion = pd.DataFrame(filas).groupby("metodo").mean()
evaluacion["precision@5"] = evaluacion["recall@5"] / 5
print(f"Clientes evaluados: {len(evaluables)} · retenidos que son recompra: {recompras}")
print(f"Recall@5 esperado al azar: {5 / (len(ids) - 2):.3f}")
evaluacion.round(3)

Clientes evaluados: 24 · retenidos que son recompra: 2
Recall@5 esperado al azar: 0.057


,recall@5,mrr,acierto_categoria@5,precision@5
metodo,,,,
aleatorio,0.042,0.021,0.583,0.008
popularidad,0.000,0.000,0.250,0.000
semantico,0.000,0.000,0.083,0.000


Cobertura del catálogo, diversidad y uso de productos sin reseñas, con el
historial completo de los 63 clientes:

In [12]:
recs_sem = {cid: recomendar(resenas[resenas["cliente_id"] == cid]) for cid in resenas["cliente_id"].unique()}
recs_pop = {cid: recomendar_populares(resenas, resenas[resenas["cliente_id"] == cid]) for cid in resenas["cliente_id"].unique()}


def resumen(recs):
    todas = [p for r in recs.values() for p in r]
    return {
        "cobertura": len(set(todas)) / len(ids),
        "categorias_distintas_top5": np.mean([len({categoria[p] for p in r}) for r in recs.values()]),
        "pct_productos_sin_resenas": np.mean([p in set(sin_resenas) for p in todas]),
    }


misma_categoria = np.mean([
    np.mean([categoria[p] in set(resenas[resenas["cliente_id"] == cid]["producto_id"].map(categoria)) for p in r])
    for cid, r in recs_sem.items()
])
print(f"Recomendaciones semánticas en una categoría ya comprada: {misma_categoria:.1%}")
pd.DataFrame({"semantico": resumen(recs_sem), "popularidad": resumen(recs_pop)}).round(3)

Recomendaciones semánticas en una categoría ya comprada: 83.5%


,semantico,popularidad
cobertura,0.867,0.067
categorias_distintas_top5,1.857,5.000
pct_productos_sin_resenas,0.403,0.000


## 5. Revisión manual de una muestra

Muestra aleatoria de 10 clientes con historial (50 recomendaciones). Se revisa
a mano si cada recomendación es coherente con el historial y si es un sustituto
de algo que el cliente ya compró.

In [13]:
muestra = random.Random(7).sample(sorted(resenas["cliente_id"].unique()), 10)
filas = []
for cid in muestra:
    h = resenas[resenas["cliente_id"] == cid]
    historial_txt = "; ".join(f"{nombre[p]} ({c}★)" for p, c in zip(h["producto_id"], h["calificacion"]))
    for rango, p in enumerate(recs_sem[cid], start=1):
        filas.append({"cliente_id": cid, "historial": historial_txt if rango == 1 else "",
                      "rango": rango, "producto_id": p, "recomendado": nombre[p], "categoria": categoria[p]})
pd.DataFrame(filas)

,cliente_id,historial,rango,producto_id,recomendado,categoria
0,37,Sennheiser Momentum 4 (2★),1,21,Sony WH-1000XM5,Audio
1,37,,2,23,Bose QuietComfort Ultra,Audio
2,37,,3,22,Apple AirPods Pro 2da Generacion,Audio
3,37,,4,49,SteelSeries Arctis Nova 7,Gaming y Consolas
4,37,,5,27,Samsung Galaxy Buds2 Pro,Audio
5,117,HP Pavilion 15 Intel Core i5 (2★); Samsung Family Hub Refrigeradora Inteligente (2★),1,13,Dell XPS 13 9340 Intel Core i7,Laptops y Computadoras
6,117,,2,20,Lenovo ThinkPad E14 Gen 5,Laptops y Computadoras
7,117,,3,36,Samsung 43 pulgadas Crystal UHD,Televisores y Video
8,117,,4,18,Acer Aspire 5 Intel Core i5,Laptops y Computadoras
9,117,,5,15,Lenovo IdeaPad Slim 3 Ryzen 5,Laptops y Computadoras


**Resultado de la revisión (50 recomendaciones):** 44 coherentes con el
historial (88 %); 39 son sustitutos de un producto ya comprado (78 %; p. ej.,
el cliente 45 compró una HP Pavilion y recibe cinco laptops); en 2 de 10
clientes domina la marca (cliente 17: Samsung The Frame → Galaxy Watch6,
Galaxy S24 Ultra, Family Hub; cliente 98: Xiaomi).

## 6. Pruebas de robustez

In [14]:
# Cliente sin historial: el prototipo no puede construir un perfil.
cid_vacio = sin_historial[0]
with np.errstate(invalid="ignore"):
    v = perfil(resenas[resenas["cliente_id"] == cid_vacio])
    lista = recomendar(resenas[resenas["cliente_id"] == cid_vacio])
print(f"Cliente {cid_vacio} sin historial -> perfil con NaN: {bool(np.isnan(v).all())}; "
      f"lista sin sentido y sin error: {lista}. El prototipo necesita un respaldo explícito.")

# Cliente con varias categorías.
h1 = resenas[resenas["cliente_id"] == 1]
print("Cliente 1, categorías del historial:", h1["producto_id"].map(categoria).value_counts().to_dict())
print("Cliente 1, categorías recomendadas:", [categoria[p] for p in recs_sem[1]])

# Producto sin reseñas: ¿aparece en las recomendaciones?
print("Sony WH-1000XM5 (id 21, sin reseñas) recomendado a:", [c for c, r in recs_sem.items() if 21 in r][:10])

# Producto sin stock: se filtra en tiempo de servido.
print("Cliente 80 con id 21 agotado:", recomendar(resenas[resenas["cliente_id"] == 80], excluir_sin_stock=[21]))

# Descripción muy corta.
q = modelo.encode(["Audifonos."], normalize_embeddings=True)[0]
print("Descripción 'Audifonos.' -> vecinos:", [nombre[ids[i]] for i in np.argsort(-(E @ q))[:5]])

Cliente 4 sin historial -> perfil con NaN: True; lista sin sentido y sin error: [1, 65, 64, 63, 62]. El prototipo necesita un respaldo explícito.
Cliente 1, categorías del historial: {'Smartphones': 4, 'Fotografia y Video': 3, 'Televisores y Video': 3}
Cliente 1, categorías recomendadas: ['Televisores y Video', 'Smartphones', 'Smartphones', 'Smartphones', 'Televisores y Video']
Sony WH-1000XM5 (id 21, sin reseñas) recomendado a: [np.int64(80), np.int64(23), np.int64(72), np.int64(43), np.int64(106), np.int64(24), np.int64(119), np.int64(37)]
Cliente 80 con id 21 agotado: [29, 28, 49, 22, 26]
Descripción 'Audifonos.' -> vecinos: ['JBL Tune 510BT', 'Apple AirPods Pro 2da Generacion', 'Samsung Galaxy Buds2 Pro', 'Bose QuietComfort Ultra', 'Sennheiser Momentum 4']


## 7. Herramientas del servidor MCP (prototipo sin LLM)

Las funciones implementan el catálogo de la sección 8: búsqueda semántica con
filtros y salidas acotadas **sin datos personales**.

In [15]:
E_resenas = modelo.encode(resenas["texto"].tolist(), normalize_embeddings=True)
E_coment = modelo.encode(comentarios["texto"].tolist(), normalize_embeddings=True)


def _top(similitud, df, filtro, campos, k):
    salida = []
    for i in np.argsort(-similitud):
        fila = df.iloc[i]
        if not filtro(fila):
            continue
        salida.append({**{c: fila[c] for c in campos}, "similitud": round(float(similitud[i]), 3)})
        if len(salida) == k:
            break
    return salida


def buscar_productos(consulta, k=5, categoria=None, precio_max=None):
    q = modelo.encode([consulta], normalize_embeddings=True)[0]
    filtro = lambda f: (precio_max is None or f["precio"] <= precio_max) and (categoria is None or f["categoria"] == categoria)
    return _top(E @ q, productos, filtro, ["id", "nombre", "categoria", "precio", "stock"], k)


def buscar_resenas_entrega(consulta, k=5, calificacion_max=None):
    q = modelo.encode([consulta], normalize_embeddings=True)[0]
    filtro = lambda f: calificacion_max is None or f["calificacion"] <= calificacion_max
    return _top(E_resenas @ q, resenas, filtro, ["id", "calificacion", "fecha", "texto"], k)


def buscar_comentarios(consulta, k=5, pais=None):
    q = modelo.encode([consulta], normalize_embeddings=True)[0]
    filtro = lambda f: pais is None or f["cliente_pais"] == pais
    return _top(E_coment @ q, comentarios, filtro, ["id", "calificacion", "fecha", "cliente_pais", "texto"], k)


HERRAMIENTAS = {f.__name__: f for f in (buscar_productos, buscar_resenas_entrega, buscar_comentarios)}

## 8. Evaluación de la recuperación (sección 9 del informe)

Los id relevantes se identificaron leyendo los CSV **antes** de ejecutar las
búsquedas. Las consultas se escriben sin tildes, como los textos de la tienda.

In [16]:
casos = [
    ("entregas con retraso sin aviso", "buscar_resenas_entrega", {}, [514, 526, 542, 557]),
    ("producto llego golpeado o rayado", "buscar_resenas_entrega", {}, [515, 525, 547, 551]),
    ("pedido incompleto, faltaba un accesorio o el cable", "buscar_resenas_entrega", {}, [516, 538, 540, 552]),
    ("audifonos con cancelacion de ruido", "buscar_productos", {"precio_max": 1000}, [22, 27]),
    ("el sitio web se pone lento o se cae en dias de alta demanda", "buscar_comentarios", {}, [329, 376, 393, 435, 439]),
]

valores_pii = (set(clientes["dni"].astype(str)) | set(clientes["email"])
               | set(clientes["nombre"] + " " + clientes["apellidos"]) | set(comentarios["cliente_nombre_completo"]))
claves_pii = {"dni", "email", "nombre", "apellidos", "cliente_id", "cliente_nombre_completo", "orden_id"}
claves_pii -= {"nombre"}  # «nombre» en productos es el nombre del producto, no de una persona

filas, fugas = [], 0
for consulta, herramienta, params, relevantes in casos:
    salida = HERRAMIENTAS[herramienta](consulta, k=5, **params)
    top = [r["id"] for r in salida]
    primero = next((i + 1 for i, x in enumerate(top) if x in relevantes), None)
    fugas += sum(bool(set(r) & claves_pii) or any(str(v) in valores_pii for v in r.values()) for r in salida)
    filas.append({"consulta": consulta, "herramienta": herramienta, "relevantes": relevantes, "top5": top,
                  "posicion_primer_relevante": primero,
                  "recall@5": len(set(top) & set(relevantes)) / len(relevantes),
                  "rr": 1 / primero if primero else 0.0})
recuperacion = pd.DataFrame(filas)
recuperacion

,consulta,herramienta,relevantes,top5,posicion_primer_relevante,recall@5,rr
0,entregas con retraso sin aviso,buscar_resenas_entrega,"[514, 526, 542, 557]","[526, 557, 535, 542, 556]",1,0.75,1.0
1,producto llego golpeado o rayado,buscar_resenas_entrega,"[515, 525, 547, 551]","[547, 553, 515, 539, 554]",1,0.50,1.0
2,"pedido incompleto, faltaba un accesorio o el cable",buscar_resenas_entrega,"[516, 538, 540, 552]","[538, 552, 492, 465, 495]",1,0.50,1.0
3,audifonos con cancelacion de ruido,buscar_productos,"[22, 27]","[27, 22, 30, 48, 26]",1,1.00,1.0
4,el sitio web se pone lento o se cae en dias de alta demanda,buscar_comentarios,"[329, 376, 393, 435, 439]","[393, 439, 396, 442, 323]",1,0.40,1.0


In [17]:
criterios = pd.DataFrame({
    "minimo": [0.80, 0.70, 0],
    "obtenido": [recuperacion["recall@5"].mean(), recuperacion["rr"].mean(), fugas],
}, index=["Recall@5 promedio", "MRR", "Respuestas con datos personales"])
criterios["cumple"] = [criterios.iloc[0, 1] >= 0.80, criterios.iloc[1, 1] >= 0.70, criterios.iloc[2, 1] == 0]
criterios.round(3)

,minimo,obtenido,cumple
Recall@5 promedio,0.8,0.63,False
MRR,0.7,1.00,True
Respuestas con datos personales,0.0,0.00,True


## Discusión

- **Recomendador:** con 24 clientes evaluables, el prototipo no acierta el
  producto retenido (recall@5 = 0), igual que las líneas base. El hallazgo
  principal es que la siguiente compra suele estar en **otra categoría**,
  mientras que el prototipo recomienda sobre todo sustitutos de lo ya comprado.
  No cumple el criterio de despliegue; la iteración siguiente debe diversificar
  (cuota por categoría o MMR) y recolectar clics en modo sombra.
- **Servidor MCP:** el primer resultado siempre es relevante (MRR = 1), pero
  el recall@5 promedio (0,63) no alcanza 0,80. Los fallos son reseñas
  relevantes con vocabulario distinto («rayon», «pantalla rota») o resultados
  positivos que comparten palabras («cable», «retraso»). Mejoras antes de
  conectar el LLM: búsqueda híbrida BM25 + vectores y filtro
  `calificacion_max` para preguntas sobre quejas.